## Codigo de Descarga Mediapipe

In [ ]:
import urllib.request
import os

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
archivo = "face_landmarker.task"

if not os.path.exists(archivo):
    print("Descargando modelo de MediaPipe...")
    urllib.request.urlretrieve(url, archivo)
    print("¡Descarga completada!")
else:
    print("El modelo ya está en la carpeta.")

Descargando modelo de MediaPipe...
¡Descarga completada!


## Filtro Integrado

In [ ]:
import cv2
import time
import os
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# Grupos de landmarks faciales
LANDMARKS_ROIS = {
    "izq":  [33, 133, 159, 145, 7, 163, 144, 153, 154, 155, 173, 157, 158, 160, 161, 246, 46, 53, 52, 65, 55, 70, 63, 105, 66, 107],
    "der":  [362, 263, 386, 374, 249, 390, 373, 380, 381, 382, 398, 384, 385, 387, 388, 466, 276, 283, 282, 295, 285, 300, 293, 334, 296, 336],
    "boca": [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 375, 321, 405, 314, 17, 84, 181, 91, 146]
}

# ==============================================================================
# CARRUSEL DE OBRAS: (escala_x, escala_y, offset_x, offset_y)
# ==============================================================================
CATALOGO_OBRAS = [
    {"nombre": "1. Mona Lisa",       "archivo": "monita.webp", "ajustes": {"izq": (1.15, 1.15, 0, 3), "der": (1.08, 1.08, 1, 5), "boca": (1.05, 1.05, 0, 0)}},
    {"nombre": "2. Frida",        "archivo": "fira.jpg",   "ajustes": {"izq": (1.12, 1.12, 0, 3), "der": (1.12, 1.12, 0, 3), "boca": (1.08, 1.08, 0, 0)}},
    {"nombre": "3. Cristobal Colon", "archivo": "colon.jpg",   "ajustes": {"izq": (1.15, 1.15, 0, 3), "der": (1.10, 1.10, 0, 2), "boca": (1.05, 1.05, 0, 0)}},
    {"nombre": "4. Van Gogh",        "archivo": "van.jpg",     "ajustes": {"izq": (1.10, 1.10, 0, 2), "der": (1.10, 1.10, 0, 2), "boca": (1.05, 1.05, 0, 0)}}
]

def obtener_bbox(puntos, indices, mx, my, max_w, max_h):
    pts = puntos[indices]
    x1, y1 = np.min(pts, axis=0)
    x2, y2 = np.max(pts, axis=0)
    return (max(2, x1 - mx), max(2, y1 - my), min(max_w - 2, x2 + mx), min(max_h - 2, y2 + my))

def ajustar_bbox(bbox, params, max_w, max_h):
    sx, sy, ox, oy = params
    x1, y1, x2, y2 = bbox
    cx, cy = ((x1 + x2) // 2) + ox, ((y1 + y2) // 2) + oy
    nw, nh = int((x2 - x1) * sx), int((y2 - y1) * sy)
    return (max(2, cx - nw // 2), max(2, cy - nh // 2), min(max_w - 2, cx + nw // 2), min(max_h - 2, cy + nh // 2))

def integrar_rasgo(fondo, recorte, bbox):
    x1, y1, x2, y2 = bbox
    w, h = x2 - x1, y2 - y1
    if w <= 8 or h <= 8 or recorte.size == 0:
        return fondo

    recorte_res = cv2.resize(recorte, (w, h), interpolation=cv2.INTER_CUBIC)
    roi_fondo = fondo[y1:y2, x1:x2]

    # Igualación estadística LAB para integrar iluminación y tono
    src_lab = cv2.cvtColor(recorte_res, cv2.COLOR_BGR2LAB).astype(np.float32)
    dst_lab = cv2.cvtColor(roi_fondo, cv2.COLOR_BGR2LAB).astype(np.float32)
    for c in range(3):
        s_mean, s_std = np.mean(src_lab[:, :, c]), np.std(src_lab[:, :, c]) + 1e-5
        d_mean, d_std = np.mean(dst_lab[:, :, c]), np.std(dst_lab[:, :, c])
        src_lab[:, :, c] = ((src_lab[:, :, c] - s_mean) * (d_std / s_std)) + d_mean

    prep = cv2.addWeighted(cv2.cvtColor(np.clip(src_lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR), 0.65, recorte_res, 0.35, 0)

    mascara = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(mascara, (w // 2, h // 2), (int(w * 0.44), int(h * 0.42)), 0, 0, 360, 255, -1)

    try:
        return cv2.seamlessClone(prep, fondo, mascara, ((x1 + x2) // 2, (y1 + y2) // 2), cv2.NORMAL_CLONE)
    except cv2.error:
        return fondo

# ==============================================================================
# 1. PRECARGA DEL CARRUSEL
# ==============================================================================
base_opts = python.BaseOptions(model_asset_path="face_landmarker.task")
opts_img = vision.FaceLandmarkerOptions(base_options=base_opts, running_mode=vision.RunningMode.IMAGE, num_faces=1)

obras = []
with vision.FaceLandmarker.create_from_options(opts_img) as detector:
    for item in CATALOGO_OBRAS:
        if not os.path.exists(item["archivo"]):
            continue
        img_raw = cv2.imread(item["archivo"])
        escala = 550.0 / img_raw.shape[0]
        img_color = cv2.resize(img_raw, (int(img_raw.shape[1] * escala), 550))
        h_o, w_o, _ = img_color.shape

        res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)))
        if not res.face_landmarks:
            continue

        pts = np.array([(int(lm.x * w_o), int(lm.y * h_o)) for lm in res.face_landmarks[0]])
        destinos = {}
        for clave, (mx, my) in [("izq", (8, 7)), ("der", (8, 7)), ("boca", (12, 10))]:
            bbox_raw = obtener_bbox(pts, LANDMARKS_ROIS[clave], mx, my, w_o, h_o)
            destinos[clave] = ajustar_bbox(bbox_raw, item["ajustes"][clave], w_o, h_o)

        obras.append({
            "nombre": item["nombre"],
            "color":  img_color,
            "dest":   destinos
        })

if not obras:
    raise RuntimeError("No se encontró ninguna imagen válida en la carpeta.")

# ==============================================================================
# 2. BUCLE EN TIEMPO REAL
# ==============================================================================
opts_vid = vision.FaceLandmarkerOptions(base_options=base_opts, running_mode=vision.RunningMode.VIDEO, num_faces=1)
cap = cv2.VideoCapture(0)
idx_obra = 0

with vision.FaceLandmarker.create_from_options(opts_vid) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        obra = obras[idx_obra]
        lienzo = obra["color"].copy()
        h_o, w_o, _ = lienzo.shape
        h_c, w_c, _ = frame.shape

        res = landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)),
            int(time.time() * 1000)
        )

        if res.face_landmarks:
            pts_cam = np.array([(int(lm.x * w_c), int(lm.y * h_c)) for lm in res.face_landmarks[0]])
            for clave, (mx, my) in [("izq", (10, 8)), ("der", (10, 8)), ("boca", (14, 12))]:
                cx1, cy1, cx2, cy2 = obtener_bbox(pts_cam, LANDMARKS_ROIS[clave], mx, my, w_c, h_c)
                lienzo = integrar_rasgo(lienzo, frame[cy1:cy2, cx1:cx2], obra["dest"][clave])

        cam_mini = cv2.resize(frame, (int(w_c * (h_o / h_c)), h_o))
        combinada = np.hstack((cam_mini, lienzo))

        hud = f"{obra['nombre']} | 'A'/'D': Cambiar Obra | ESC: Salir"
        cv2.putText(combinada, hud, (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 180), 2, cv2.LINE_AA)
        cv2.imshow("Museo Vivo", combinada)

        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break
        elif key in (ord('d'), ord('D'), 32):
            idx_obra = (idx_obra + 1) % len(obras)
        elif key in (ord('a'), ord('A')):
            idx_obra = (idx_obra - 1) % len(obras)

cap.release()
cv2.destroyAllWindows()